In [1]:
import pandas as pd


In [2]:
#통합 데이터 준비(merge :합치기)
# products, orders, order_items


prod = pd.read_csv('../data/products.csv')
orders = pd.read_csv('../data/orders.csv')
oi = pd.read_csv('../data/order_items.csv')

In [3]:
#중복된 상품번호가 있는지 확인
# prod['product_id'].nunique() #497   

# prod.info()   #3개는 중복인걸 확인한거임
prod = prod.drop_duplicates('product_id')

# prod.info()

In [4]:
#category 문제 확인 : 공백이 존재하는게 문제  => 삭제 

prod['category'] = prod['category'].str.strip()

In [5]:
#카테고리별 매출 합계
orders.head()

,order_id,customer_id,order_datetime,channel,status
0,25463,1077,2024-06-25 00:17:41,store,delivered
1,171088,2998,2024-05-28 19:35:20,web,delivered
2,27437,4066,2024-02-14 17:49:14,app,delivered
3,98425,3243,2024-05-08 16:37:36,store,delivered
4,22325,5331,2024-04-17 20:08:22,app,delivered


In [20]:
#order_items
oi.head()     #order_item_id 는 주문 상세 번호

#1. 주문 상세번호 중복 여부 확인
# oi.info()  #unit_price NaN
# oi.describe()  #quantity -2
oi = oi.drop_duplicates('order_item_id')  #row 단위 삭제 

#2. 단가: NaN 여부를 확인
#subset에 null 이 많은 unit_price를 넣어야됨. 하지만 []  <- meaning 여러개 들어갈 수 있다
oi = oi.dropna(subset=['unit_price'])

#3. 수량 : 0 이거나 음수
oi = oi[oi['quantity']>0]

#4. 전체 주문 금액
# vector 처리
oi['amount'] = oi['unit_price'] * oi['quantity'] * (1 - oi['discount'])

In [7]:
oi.describe()

,order_item_id,order_id,product_id,quantity,unit_price,discount,amount
count,484287.000000,484287.000000,484287.000000,484287.000000,484287.000000,484287.000000,484287.000000
mean,249915.038033,99960.579425,220.309217,3.001212,50926.766566,0.249773,50924.515364
std,144300.108250,57753.899340,150.595851,1.412893,48910.628653,0.144480,48910.629783
min,1.000000,1.000000,1.000000,1.000000,1000.000000,0.000000,995.000000
25%,124933.500000,49942.500000,92.000000,2.000000,15100.000000,0.120000,15098.380000
50%,249887.000000,99927.000000,216.000000,3.000000,27000.000000,0.250000,26999.400000
75%,374914.500000,149944.500000,348.000000,4.000000,73700.000000,0.370000,73695.550000
max,499880.000000,200000.000000,497.000000,5.000000,291000.000000,0.500000,290999.490000


In [8]:
# 조인

oi.info()

<class 'pandas.DataFrame'>
Index: 484287 entries, 0 to 499999
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  484287 non-null  int64  
 1   order_id       484287 non-null  int64  
 2   product_id     484287 non-null  int64  
 3   quantity       484287 non-null  int64  
 4   unit_price     484287 non-null  float64
 5   discount       484287 non-null  float64
 6   amount         484287 non-null  float64
dtypes: float64(3), int64(4)
memory usage: 29.6 MB


In [66]:
orders.info()

#oi 와 orders를 order_id로 연결

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   order_id        200000 non-null  int64
 1   customer_id     200000 non-null  int64
 2   order_datetime  198067 non-null  str  
 3   channel         200000 non-null  str  
 4   status          200000 non-null  str  
dtypes: int64(2), str(3)
memory usage: 13.4 MB


In [21]:
#1. orders, oi 를 join => join 조건 orders_id
# merge() method
# innerjoin은 일치하는것
# 왼쪽데이터.merge(오른쪽데이터, 합치는_조건)

#oi 와 orders가 join가 되는거임
#복사해서 하기때문에 원본은 바뀌지 않는다 -> 대입
m = oi.merge(orders, how = 'inner', on= 'order_id')

In [22]:
m.head()

,order_item_id,order_id,product_id,quantity,unit_price,discount,amount,customer_id,order_datetime,channel,status
0,59256,114625,3,1,"20,300",0,"19,285",5614,2024-06-13 16:58:05,app,delivered
1,230587,66337,80,2,"90,000",0,"99,000",5500,2024-04-02 18:18:21,app,paid
2,279813,163343,3,3,"20,300",0,"46,893",4823,2024-06-26 06:42:37,store,delivered
3,88487,180332,79,1,"8,600",0,"5,332",4266,2024-02-02 08:59:27,app,delivered
4,39240,174179,232,1,"42,600",0,"29,394",5564,2024-06-22 06:41:40,web,delivered


In [69]:
prod.head()
#m과 prod 를 product_id로 inner join

,product_id,product_name,category,price,cost
0,119,플러스 홍차,식품,0,3700.0
1,95,프리미엄 노트북,전자,68600.0,28900.0
2,112,프리미엄 원두커피,식품,10500.0,4700.0
3,164,베이직 마우스,전자,52100.0,32600.0
4,274,베이직 홍차,식품,16000.0,10500.0


In [23]:
m = m.merge(prod, on='product_id')
m.head()

,order_item_id,order_id,product_id,quantity,unit_price,discount,amount,customer_id,order_datetime,channel,status,product_name,category,price,cost
0,59256,114625,3,1,"20,300",0,"19,285",5614,2024-06-13 16:58:05,app,delivered,플러스 잡지,도서,20300.0,"10,500"
1,230587,66337,80,2,"90,000",0,"99,000",5500,2024-04-02 18:18:21,app,paid,플러스 선반,가구,90000.0,"65,700"
2,279813,163343,3,3,"20,300",0,"46,893",4823,2024-06-26 06:42:37,store,delivered,플러스 잡지,도서,20300.0,"10,500"
3,88487,180332,79,1,"8,600",0,"5,332",4266,2024-02-02 08:59:27,app,delivered,플러스 립스틱,뷰티,8600.0,"5,900"
4,39240,174179,232,1,"42,600",0,"29,394",5564,2024-06-22 06:41:40,web,delivered,프리미엄 립스틱,뷰티,42600.0,"31,400"


In [12]:
m['category'].unique()

<ArrowStringArray>
['도서', '가구', '뷰티', '전자', '의류', '식품']
Length: 6, dtype: str

In [24]:
#sort_values
m.groupby('category')['amount'].sum().sort_values(ascending=False)

category
전자   23,706,632,357
가구   16,883,450,865
의류    4,773,475,968
도서    4,550,701,174
뷰티    3,627,218,386
식품    1,953,853,982
Name: amount, dtype: float64

In [25]:
# 목표: 카테고리별 매출, 건수,   => 평균단가
m.groupby('category') #category로 grouping => DataFrameGroupBy(object = > method) 
#agg() method 사용 가능
#agg() : 집계(합, 개수 , 평균)
#agg() 는 새로운 DataFrame을 만든다

report=m.groupby('category').agg(
    sales=('amount', 'sum'),  #카테고리별 매출
    orders =('order_item_id', 'count'), #카테고리별 개수
    avg_price=('unit_price', 'mean') #카테고리별 평균단가
)

In [94]:
report

,sales,orders,avg_price
category,,,
가구,"7,518,479,149",59089,"127,242"
도서,"2,021,420,080",133445,"15,150"
뷰티,"1,613,541,309",62009,"26,023"
식품,"867,816,182",62371,"13,916"
의류,"2,115,620,687",58637,"36,082"
전자,"10,525,203,366",108736,"96,798"


In [ ]:
#카테고리별 매출 비중
#transform은 집계값을 원래 행에 되돌릴 때 쓴다

In [26]:
m['cate_total'] = m.groupby('category')['amount'].transform('sum')
# m.head()
m['share'] = m['amount'] / m['cate_total']
m[['category', 'amount', 'cate_total', 'share']]        


,category,amount,cate_total,share
0,도서,"19,285","4,550,701,174",0
1,가구,"99,000","16,883,450,865",0
2,도서,"46,893","4,550,701,174",0
3,뷰티,"5,332","3,627,218,386",0
4,뷰티,"29,394","3,627,218,386",0
...,...,...,...,...
484282,도서,"49,532","4,550,701,174",0
484283,의류,"213,345","4,773,475,968",0
484284,뷰티,"8,580","3,627,218,386",0
484285,뷰티,"149,850","3,627,218,386",0


In [16]:

# orders['channel'].value_counts()
# channel
# web      109958
# app       59894
# store     30148


orders['status'].value_counts()
# status
# delivered    109976
# shipped       30017
# paid          29993
# canceled      20039
# returned       9975


status
delivered    109976
shipped       30017
paid          29993
canceled      20039
returned       9975
Name: count, dtype: int64

In [27]:
# 채널, 상태별 개수
combo = orders.groupby(['channel', 'status']).size()
combo

# multi index -> index만들어서 바로 선택할 수 있게 하는것
# 다중 key 그룹화 -> pivot table
#pivot table: aggregation

# channel  status   
# app      canceled      5865
#          delivered    32845
#          paid          9169
#          returned      2975
#          shipped       9040
# store    canceled      3002
#          delivered    16676
#          paid          4460
#          returned      1570
#          shipped       4440
# web      canceled     11172
#          delivered    60455
#          paid         16364
#          returned      5430
#          shipped      16537

channel  status   
app      canceled      5865
         delivered    32845
         paid          9169
         returned      2975
         shipped       9040
store    canceled      3002
         delivered    16676
         paid          4460
         returned      1570
         shipped       4440
web      canceled     11172
         delivered    60455
         paid         16364
         returned      5430
         shipped      16537
dtype: int64

In [ ]:
#pivot table 
#3개의 column들을 각각 index, columns, values에 넣어서 새로운 pivot table 을 만듬
#aggfunc  -> category별 channel별 amount의 합

In [28]:
# 카테고리, 주문 경로별 매출합
# pt pivot table
pd.set_option('display.float_format', '{:,.0f}'.format)

pt = pd.pivot_table(
    data = m,           #m : dataframe
    index = 'category',  #m에 있는 category 컬럼을 추출, index로 보내겠다
    columns = 'channel', #m 에 있는  channel 컬럼을 column(열) 로 보내겠다
    values= 'amount',
    aggfunc = 'sum',
    margins=True,
    margins_name= "합",
    fill_value=0         #NaN인 값을 0으로 대체
)
pt


channel,app,store,web,합
category,,,,
가구,"5,011,512,370","2,625,032,543","9,246,905,952","16,883,450,865"
도서,"1,355,128,274","682,645,063","2,512,927,837","4,550,701,174"
뷰티,"1,097,429,571","535,683,843","1,994,104,972","3,627,218,386"
식품,"590,618,118","292,373,904","1,070,861,960","1,953,853,982"
의류,"1,444,389,650","712,135,574","2,616,950,744","4,773,475,968"
전자,"7,077,176,119","3,588,209,569","13,041,246,669","23,706,632,357"
합,"16,576,254,102","8,436,080,496","30,482,998,134","55,495,332,732"


In [29]:
multi = pd.pivot_table(
    data=m,
    index='category',
    values='amount',
    aggfunc=['sum','mean','count']
)
multi

,sum,mean,count
,amount,amount,amount
category,,,
가구,"16,883,450,865","285,729",59089
도서,"4,550,701,174","34,102",133445
뷰티,"3,627,218,386","58,495",62009
식품,"1,953,853,982","31,326",62371
의류,"4,773,475,968","81,407",58637
전자,"23,706,632,357","218,020",108736
